# Thermo-optic phase shifter: TiN heater over a silicon strip

How much heater power shifts the phase of a silicon waveguide by π, and how fast does the heater switch? This notebook
answers both for the TiN-over-strip heater of Jacques et al., *Opt. Express* **27**, 10456 (2019),
[doi:10.1364/OE.27.010456](https://doi.org/10.1364/OE.27.010456), on 220 nm SOI. It checks gsim two ways:

- **against the paper's own simulations**, with the same cross-section and the same boundary conditions;
- **against the paper's measurements** of the heater it fabricated.

[`gsim.thermal`](../api/thermal.md) solves steady and transient heat conduction in a 2D cross-section of an infinitely
long heater. It then re-solves the waveguide's optical mode with the heated refractive index. This gives P_π, the
switching time, the heater resistance and the crosstalk onto a neighbouring waveguide.

**Requirements:** the `femwell` extra (`pip install "gsim[femwell]"`). Everything runs locally in a few minutes, with no
cloud job.

**Units:** lengths in µm, heater power in mW (total over the heater length), temperature as a rise above ambient in K,
time in µs. Material properties are in SI units.

## What the paper reports

The paper simulates TiN heaters of several widths with Lumerical DEVICE, then fabricates one of them. Its numbers
belong to three different heaters, and only some of them are measurements:

| Quantity | Heater width | Value | Kind | Where |
| --- | --- | --- | --- | --- |
| P_π | 2.0 µm | 21.4 mW and 23.9 mW | simulated, two values | abstract and §3.1; §2.2 |
| τ, 1/e | 2.5 µm | 5.6 µs | simulated | abstract, §3.2 |
| P_π | 7.5 µm (fabricated) | 29.2–31.5 mW | measured, several MZIs | §3.1 |
| τ, 1/e rise / fall | 7.5 µm (fabricated) | 3.7 / 7.2 µs | measured, oscilloscope | §3.2 |
| R | 7.5 µm (fabricated) | 0.54 kΩ, total | measured, includes contacts | §3.1 |

Three points matter for the comparison:

- **The paper gives two simulated P_π for the 2.0 µm heater.** The abstract's "optimum TiN design has a switching
  power P_π = 21.4 mW", and §3.1 says the 2.0 µm heater, "the most efficient in simulations, could achieve a ~21 mW
  P_π". §2.2 extracts 23.9 mW from the 40 × 18 µm Lumerical model. The paper does not reconcile the two, and it has
  other small slips: its N++ heater is 25.2 mW in Fig. 1 and 25.5 mW in §2.2. So the target here is the range,
  21.4–23.9 mW. The abstract's τ = 5.6 µs belongs to a different design, the fastest (2.5 µm wide). Neither design
  was fabricated.
- **τ is a 1/e time.** The paper defines it as "the 1/e limiting time constant of heat diffusion (rise or fall)". So
  `tau_1e` is the comparable number. The 10–90 % time is shown too, but only for reference.
- **There is no V_π for this device**, and the paper withholds the TiN and oxide thicknesses (Table 3).

## Inputs: what comes from the paper and what we assume

| Input | Value | Source |
| --- | --- | --- |
| Si strip | 0.50 × 0.22 µm | paper |
| Buried oxide (BOX) | 2.0 µm | paper |
| Oxide gap, Si top to heater bottom | 2.0 µm | paper ("located 2 µm above the Si strip") |
| TiN heater | 2.0, 2.5 or 7.5 µm wide, 320 µm long | paper |
| TiN thickness | **0.14 µm** | withheld; femwell's reproduction. 0.20 µm (Tidy3D's) shown as a sensitivity |
| Oxide above the heater | **0.5 µm** | withheld; assumption. 0.08 µm (Tidy3D's) shown as a sensitivity |
| k: Si strip / Si substrate / SiO2 / TiN (W/m/K) | 90 / 148 / 1.38 / 28 | paper Table 3; 90 is the 220 nm film value |
| ρ c_p: Si / SiO2 / TiN | gsim's built-in values | the same as Table 3 as transcribed by Tidy3D |
| TiN conductivity | σ = 2.3e6 S/m (43 µΩ·cm) | paper Table 3; it sets R only |
| dn/dT: Si / SiO2 (1/K) | 1.86e-4 / 1e-5 | femwell; gsim's built-in Si value is the same |
| Thermal region | 40 µm wide × 18 µm high, bottom in the Si substrate at ambient | paper §2.2 |
| Top of the oxide | convective, h = 10 W/m²/K | paper §2.2 |
| Sides of the region | adiabatic | assumption; the paper does not say |

In [ ]:
import math
import warnings

import gdsfactory as gf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gsim.common.stack import MATERIALS_DB, Layer, LayerStack
from gsim import thermal

gf.gpdk.PDK.activate()

WAVELENGTH = 1.55  # um
LENGTH = 320.0  # heater length, um
CORE_W, CORE_T = 0.50, 0.22  # Si strip, um
BOX = 2.0  # buried oxide, um
GAP = 2.0  # Si top to heater bottom, um
HEATER_T = 0.14  # TiN thickness (withheld in the paper), um
CLAD_ABOVE = 0.5  # oxide above the heater (withheld in the paper), um
REGION_W, REGION_H = 40.0, 18.0  # paper's thermal simulation region, um

# Jacques et al. 2019: simulated (Lumerical) and measured reference values
P_PI_SIM_2U0 = (21.4, 23.9)  # mW, 2.0 um TiN, simulated: abstract, Sec. 2.2
TAU_SIM_2U5 = 5.6  # us, 2.5 um TiN, simulated, 1/e
P_PI_MEAS_7U5 = (29.2, 31.5)  # mW, 7.5 um TiN, measured
TAU_RISE_7U5, TAU_FALL_7U5 = 3.7, 7.2  # us, 7.5 um TiN, measured, 1/e
R_MEAS_7U5 = 540.0  # ohm, 7.5 um TiN, measured total including contacts


def outside(value, band):
    # Relative distance of value outside the band (lo, hi); 0 inside.
    lo, hi = band
    return min(0.0, value / lo - 1) + max(0.0, value / hi - 1)


def describe(error):
    return "inside" if error == 0 else f"{error:+.1%} outside"

## Geometry

The device is a straight Si strip under a TiN strip. gdsfactory's `strip_heater_metal` cross-section draws both: the
waveguide on `WG` (1/0) and the heater on `HEATER` (47/0). We cut it at mid-length, `x = 160 µm`, so the in-plane axis of
the cross-section is `y`.

In [ ]:
def heater_over_strip(heater_w, neighbour_gap=None):
    # TiN heater over a Si strip, optionally with a second strip at neighbour_gap um.
    xs = gf.cross_section.strip_heater_metal(width=CORE_W, heater_width=heater_w)
    c = gf.Component()
    c << gf.components.straight(length=LENGTH, cross_section=xs)
    if neighbour_gap is not None:
        victim = c << gf.components.straight(length=LENGTH, width=CORE_W)
        victim.dmovey(neighbour_gap)
    return c


heater_over_strip(2.0).plot()

## Layer stack

The stack gives each layer its height and material. The Si strip and the Si substrate are both silicon, but they conduct
heat differently. A 220 nm film conducts about 90 W/m/K, and bulk silicon 148 W/m/K. `sim.materials`
overrides a property for every layer made of a given material. So the strip gets its own material name, `si_film`, with
silicon's optical record, and the override then reaches only the strip.

The paper's thermal region is 18 µm high, from a point in the Si substrate up to the top of the oxide. With our assumed
0.5 µm of oxide above a 0.14 µm heater, the oxide top sits 4.86 µm above the substrate. The rest of the 18 µm is Si
substrate: that is the Modelled depth, and its bottom is held at ambient.

In [ ]:
heater_z0 = CORE_T + GAP


def oxide_top(heater_t=HEATER_T, clad_above=CLAD_ABOVE):
    # Top of the oxide above the BOX top (z = 0), um.
    return heater_z0 + heater_t + clad_above


def region_depth(heater_t=HEATER_T, clad_above=CLAD_ABOVE):
    # Si substrate depth that makes the thermal region REGION_H high, um.
    return REGION_H - BOX - oxide_top(heater_t, clad_above)


def jacques_stack(heater_t=HEATER_T, clad_above=CLAD_ABOVE, substrate=20.0):
    # Paper cross-section: z = 0 is the BOX top, the Si substrate is the heat sink.
    return LayerStack(
        pdk_name="jacques2019",
        layers={
            "core": Layer(
                name="core",
                gds_layer=(1, 0),
                zmin=0.0,
                zmax=CORE_T,
                thickness=CORE_T,
                material="si_film",
                layer_type="dielectric",
            ),
            "heater": Layer(
                name="heater",
                gds_layer=(47, 0),
                zmin=heater_z0,
                zmax=heater_z0 + heater_t,
                thickness=heater_t,
                material="TiN",
                layer_type="conductor",
            ),
        },
        # Si film: silicon's optical and thermal record, under its own name.
        materials={"si_film": MATERIALS_DB["silicon"].to_dict()},
        dielectrics=[
            {
                "name": "substrate",
                "zmin": -BOX - substrate,
                "zmax": -BOX,
                "material": "silicon",
            },
            {"name": "box", "zmin": -BOX, "zmax": 0.0, "material": "SiO2"},
            {
                "name": "clad",
                "zmin": 0.0,
                "zmax": oxide_top(heater_t, clad_above),
                "material": "SiO2",
            },
        ],
    )


print(f"oxide top: {oxide_top():.2f} um above the BOX top")
print(
    f"Modelled depth for an {REGION_H:g} um high region: {region_depth():.2f} um of Si"
)

## Simulation setup

`make_sim` reproduces the paper's set-up:

- the **Window**, the in-plane extent of the cross-section, spans ±20 µm (40 µm wide); its sides are adiabatic;
- the **Modelled depth** makes the region 18 µm high, with the substrate bottom held at ambient;
- the top of the oxide loses heat by convection, h = 10 W/m²/K.

The first waveguide added is the phase shifter, and any later one is a crosstalk victim.

In [ ]:
def make_sim(
    heater_w=2.0,
    heater_t=HEATER_T,
    clad_above=CLAD_ABOVE,
    window=(-REGION_W / 2, REGION_W / 2),
    depth=None,
    top="convective",
    neighbour_gap=None,
    precision="default",
):
    depth = region_depth(heater_t, clad_above) if depth is None else depth
    sim = thermal.Simulation()
    sim.geometry(
        heater_over_strip(heater_w, neighbour_gap),
        stack=jacques_stack(heater_t, clad_above, substrate=max(depth, 20.0)),
    )
    sim.domain(plane="x=160", window=window, depth_um=depth)
    sim.heater(layers="heater", length_um=LENGTH)
    sim.boundary(top=top, heat_transfer_coefficient=10.0)
    # Thin-film Si conductivity, paper's TiN conductivity, femwell's oxide dn/dT
    sim.materials["si_film"] = {"thermal_conductivity": 90.0}
    sim.materials["TiN"] = {"conductivity": 2.3e6}
    sim.materials["SiO2"] = {"dn_dT": 1e-5}
    sim.waveguides.add(name="arm", position_um=0.0)
    if neighbour_gap is not None:
        sim.waveguides.add(name="neighbour", position_um=neighbour_gap)
    sim.solver(precision=precision)
    return sim

## Simulation against simulation: the paper's Lumerical model

### P_π of the 2.0 µm heater

`solve` runs one heat solve. The model is linear in power, so that one solve gives the temperature rise at every power.
It then re-solves the waveguide's mode with the heated index to get the effective-index shift per milliwatt. P_π is
the power at which that shift adds up to a phase of π over the 320 µm heater.

In [ ]:
sim_2u0 = make_sim(heater_w=2.0)
result_2u0 = sim_2u0.solve(wavelength_um=WAVELENGTH)
print(
    f"P_pi = {result_2u0.p_pi_mw:.2f} mW, efficiency = {result_2u0.efficiency_rad_per_mw:.4f} rad/mW"
)
print(f"Core rise per mW = {result_2u0.thermal_resistance_k_per_mw['arm']:.3f} K/mW")
lo, hi = P_PI_SIM_2U0
print(
    f"Lumerical {lo}-{hi} mW: gsim is {describe(outside(result_2u0.p_pi_mw, P_PI_SIM_2U0))}"
    f" ({result_2u0.p_pi_mw / lo - 1:+.1%} / {result_2u0.p_pi_mw / hi - 1:+.1%} from the ends)"
)

In [ ]:
tmap = result_2u0.temperature(power_mw=result_2u0.p_pi_mw)
top_z = oxide_top()

fig, (ax_map, ax_cut) = plt.subplots(
    1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2, 1]}
)
tmap.plot(ax=ax_map)
ax_map.set_xlim(-8, 8)
ax_map.set_ylim(-4, top_z)

z = np.linspace(-BOX - region_depth() + 0.01, top_z - 0.01, 400)
ax_cut.plot(tmap.at(0.0, z), z)
for z_line, label in [(0.0, "BOX top"), (-BOX, "substrate top"), (heater_z0, "heater")]:
    ax_cut.axhline(z_line, color="0.6", lw=0.8, ls=":")
    ax_cut.text(0.5, z_line + 0.15, label, fontsize=8, color="0.4")
ax_cut.set_xlabel("Temperature rise at y = 0 (K)")
ax_cut.set_ylabel("z (µm)")
ax_cut.set_title("Vertical cut through the heater")
fig.tight_layout()

for z_probe, label in [
    (heater_z0 + HEATER_T / 2, "heater centre"),
    (CORE_T / 2, "core centre"),
    (-BOX, "substrate top"),
]:
    print(
        f"rise at {label:14s} (z = {z_probe:5.2f} um): {tmap.at(0.0, z_probe):6.2f} K"
    )

Almost all of the temperature drop is across the oxide. The silicon substrate, 100 times more conductive, stays within
a few tenths of a kelvin of ambient. That is why P_π depends on the oxide's conductivity and thickness much more than on
the heater or the silicon.

As a sanity check we also use a closed form. It treats the heater as a strip of line sources in uniform oxide, H = 4.86
µm thick, with an adiabatic top and an ideal heat sink below (method of images). It evaluates the rise at the core
centre and takes ΔT_π = λ / (2 L dn/dT_Si). That closed form ignores the Si core, the mode's spread and the substrate, so
it agrees only to about 10 %.

In [ ]:
def strip_over_sink_rise(H, depth_heater, depth_point, width, k, n=401):
    # Rise (K per W/m) at a point on the axis under a strip heater in a uniform layer
    # with an adiabatic top and an ideal sink at depth H (method of images).
    def line(x, y, y0):
        c = np.cosh(np.pi * x / (2 * H))
        return np.log(
            (c + np.cos(np.pi * (y + y0) / (2 * H)))
            / (c - np.cos(np.pi * (y - y0) / (2 * H)))
        )

    xs = np.linspace(-width / 2, width / 2, n)
    g = line(xs, depth_point, depth_heater) + line(xs, depth_point, -depth_heater)
    return float(np.mean(g)) / (4 * np.pi * k)


H_OXIDE = oxide_top() + BOX  # adiabatic top to the substrate, um
rise_per_w_per_m = strip_over_sink_rise(
    H_OXIDE, CLAD_ABOVE + HEATER_T / 2, top_z - CORE_T / 2, 2.0, k=1.38
)
dT_pi = WAVELENGTH / (2 * LENGTH * 1.86e-4)  # K
p_pi_closed_form = dT_pi / rise_per_w_per_m * LENGTH * 1e-6 * 1e3  # mW

references = [*P_PI_SIM_2U0, p_pi_closed_form]
pd.DataFrame(
    {
        "P_pi (mW)": references,
        "gsim vs it": [f"{result_2u0.p_pi_mw / p - 1:+.1%}" for p in references],
    },
    index=[
        "Lumerical (Jacques 2019, abstract)",
        "Lumerical (Jacques 2019, §2.2)",
        "closed form, uniform oxide",
    ],
).round(2)

### τ of the 2.5 µm heater

`solve_transient` switches the heater on at t = 0 and follows the phase until it settles. It reports two times:

- `tau_1e`, the time to reach 1 − 1/e (63 %) of the final phase: the paper's definition;
- `tau`, the 10–90 % rise time, shown for reference.

In [ ]:
sim_2u5 = make_sim(heater_w=2.5)
result_2u5 = sim_2u5.solve(wavelength_um=WAVELENGTH)
step_2u5 = sim_2u5.solve_transient(
    power_mw=result_2u5.p_pi_mw, duration_us=100.0, wavelength_um=WAVELENGTH
)

ax = step_2u5.plot()
ax.axvline(TAU_SIM_2U5, color="C3", ls="-.", label=f"Lumerical tau = {TAU_SIM_2U5} µs")
ax.legend()
print(f"P_pi (2.5 um) = {result_2u5.p_pi_mw:.2f} mW")
print(
    f"tau_1e = {step_2u5.tau_1e_us:.2f} us vs Lumerical {TAU_SIM_2U5} us: "
    f"{step_2u5.tau_1e_us / TAU_SIM_2U5 - 1:+.0%} (target +/-20 %)"
)
print(f"tau (10-90 %) = {step_2u5.tau_us:.2f} us, for reference")

The time step and the mesh are not the cause of any gap: the same step with `precision="fast"` (a coarser mesh and half
the time steps) gives nearly the same times.

In [ ]:
sim_2u5.solver(precision="fast")
step_2u5_fast = sim_2u5.solve_transient(
    power_mw=result_2u5.p_pi_mw, duration_us=100.0, wavelength_um=WAVELENGTH
)
print(f'precision="fast":    tau_1e = {step_2u5_fast.tau_1e_us:.2f} us')
print(f'precision="default": tau_1e = {step_2u5.tau_1e_us:.2f} us')

### The withheld thicknesses

The paper withholds the TiN and oxide thicknesses. The table repeats both comparisons with Tidy3D's values in place of
ours, one at a time. These rows use `precision="fast"` and are compared with a fast baseline: the fast and default
solves differ by well under 1 % in P_π and about 2 % in τ, which does not change any conclusion here.

In [ ]:
def p_pi_and_tau(**kwargs):
    # P_pi of the 2.0 um heater and tau_1e of the 2.5 um heater, fast precision.
    p_pi = (
        make_sim(heater_w=2.0, precision="fast", **kwargs)
        .solve(wavelength_um=WAVELENGTH)
        .p_pi_mw
    )
    sim = make_sim(heater_w=2.5, precision="fast", **kwargs)
    power = sim.solve(wavelength_um=WAVELENGTH).p_pi_mw
    tau_1e = sim.solve_transient(
        power_mw=power, duration_us=100.0, wavelength_um=WAVELENGTH
    ).tau_1e_us
    return p_pi, tau_1e


variants = {
    "assumed: TiN 0.14 µm, oxide above 0.5 µm": {},
    "TiN 0.20 µm": {"heater_t": 0.20},
    "oxide above the heater 0.08 µm": {"clad_above": 0.08},
}
rows = []
for label, kwargs in variants.items():
    p_pi, tau_1e = p_pi_and_tau(**kwargs)
    rows.append(
        {
            "inputs": label,
            "P_pi, 2.0 µm (mW)": round(p_pi, 2),
            "vs 21.4–23.9 mW": describe(outside(p_pi, P_PI_SIM_2U0)),
            "tau_1e, 2.5 µm (µs)": round(tau_1e, 2),
            "vs 5.6 µs": f"{tau_1e / TAU_SIM_2U5 - 1:+.0%}",
        }
    )
pd.DataFrame(rows).set_index("inputs")

**P_π falls inside the paper's simulated range; τ does not.**

- **P_π** of the 2.0 µm heater lands between the paper's two simulated values: 5.7 % above 21.4 mW and 5.3 % below
  23.9 mW. Refining the mesh moves it by a few tenths of a percent (see the convergence table below), so the
  numerics cannot pick between the two. The withheld thicknesses move it more. A 0.20 µm heater raises P_π by about
  1 %. The withheld oxide above the heater matters more: 0.08 µm lowers P_π by 7 %, to just under the range, because
  less heat then spreads sideways through the cladding above the heater and more of it flows down past the core.
  Neither input is tuned here.
- **τ** of the 2.5 µm heater comes out about 58 % above the paper's 5.6 µs, well outside ±20 %. The thin top oxide
  shortens it by about 15 %, still far outside. The numerics are not the cause: the fast and default solves agree, and the response
  has one dominant time constant (the 10–90 % to 1/e ratio is about 2.15, against 2.2 for a single exponential).

The physics bounds τ from below. Heat diffusing through the 4.86 µm of oxide has a slowest 1D time constant of
4ρc_pH² / (π²k) ≈ 11 µs, with the paper's Table 3 values for the oxide (adiabatic top, sink below). The model's 1/e
time sits under that, as it should, because the core is closer to the heater than the sink is. Reaching 5.6 µs needs
an oxide ρc_p/k about a third lower than Table 3's, or a much thinner stack than the paper describes. Likely reasons for
the gap, roughly in order:

1. **How the simulated τ was extracted.** The paper does not say whether its 5.6 µs is a 1/e crossing, a fit to the
   early response, or a time taken at the heater rather than at the waveguide. The latter two read short.
2. **Heat capacities.** We use Table 3's ρ c_p as transcribed by Tidy3D. If Lumerical's material defaults differed, τ
   scales with them.
3. **Unstated set-up details**, such as where in the 18 µm region the time constant was taken, or the thickness of the
   oxide above the heater.

**Two independent tools side with the model, not with 5.6 µs.** On the same mesh, with the same materials and
source, femwell's own transient solver gives the same temperature history as gsim to better than 1e-9 K (this is a
test in `tests/thermal/test_femwell_crosscheck.py`). Tidy3D's transient tutorial for this paper's TiN heater
(2.0 × 0.2 µm heater, 0.08 µm oxide above it, 40 × 18 µm region, Table 3 materials) plots the core temperature crossing
1 − 1/e at about 8 µs, read off its figure; gsim with that exact set-up gives 7.7 µs. Neither reproduces the paper's
5.6 µs, so that number is a doubtful target rather than evidence of a gsim error.

## Simulation against measurement: the fabricated 7.5 µm heater

The paper fabricated "the TiN design with a 7.5 µm wide resistor (widest simulated in Fig. 2)". The same set-up gives
its P_π, its switching time and its resistance.

In [ ]:
sim_7u5 = make_sim(heater_w=7.5)
result_7u5 = sim_7u5.solve(wavelength_um=WAVELENGTH)
step_7u5 = sim_7u5.solve_transient(
    power_mw=result_7u5.p_pi_mw, duration_us=100.0, wavelength_um=WAVELENGTH
)

low, high = P_PI_MEAS_7U5
band = (0.85 * low, 1.15 * high)
print(
    f"P_pi = {result_7u5.p_pi_mw:.2f} mW; measured {low}-{high} mW; "
    f"+/-15 % band {band[0]:.1f}-{band[1]:.1f} mW"
)
print(
    f"tau_1e = {step_7u5.tau_1e_us:.2f} us; measured 1/e rise {TAU_RISE_7U5} us, fall {TAU_FALL_7U5} us"
)
print(f"tau (10-90 %) = {step_7u5.tau_us:.2f} us, for reference")
print(f"R = {result_7u5.resistance_ohm:.0f} ohm; measured {R_MEAS_7U5:.0f} ohm total")

In [ ]:
t = step_7u5.time_us
fig, ax = plt.subplots()
ax.plot(t, step_7u5.phase_rad / step_7u5.steady_phase_rad, label="gsim, 7.5 µm heater")
ax.plot(
    t,
    1 - np.exp(-t / TAU_RISE_7U5),
    "--",
    label=f"measured rise, τ = {TAU_RISE_7U5} µs",
)
ax.plot(
    t, 1 - np.exp(-t / TAU_FALL_7U5), ":", label=f"measured fall, τ = {TAU_FALL_7U5} µs"
)
ax.axhline(1 - 1 / math.e, color="0.7", lw=0.8)
ax.set_xlim(0, 40)
ax.set_xlabel("Time after the switch (µs)")
ax.set_ylabel("Fraction of the final phase change")
ax.set_title("Switching: model against the paper's 1/e fits")
ax.legend()
fig.tight_layout()

**P_π agrees with the measurement; the model switches slower than the device.**

- **P_π** of the 7.5 µm heater falls inside the measured 29.2–31.5 mW, well within the ±15 % band. The paper quotes
  its own simulation as 3.0 mW from the measurement for this heater.
- **τ.** The linear model has one time constant for rising and falling: the fall is the mirror image of the rise. The
  measured rise (3.7 µs) and fall (7.2 µs) differ by a factor of two, which a linear 2D model cannot reproduce. The
  model's single `tau_1e` lands above both, 42 % above the fall. That is the same direction as the gap to the paper's
  simulated 5.6 µs, so the likely cause is at least partly shared (see above) rather than specific to the measurement. The
  rise/fall asymmetry itself is outside this model. Likely sources:
  - the oscilloscope records the MZI's optical output, which goes as sin²(φ/2), not the phase φ, so the 1/e points of
    the trace sit at different phase fractions on the rise and the fall;
  - temperature-dependent properties and the heater's own resistance, which rises as it heats under a voltage drive;
  - 3D heat flow at the heater ends, contacts and routing.
- **R** is reported, not compared. The model gives the resistance of the bare TiN strip with the paper's conductivity
  and our assumed thickness. The measured 0.54 kΩ is a total that includes contacts and routing, and the TiN thickness
  is withheld. If the whole 0.54 kΩ were in the strip, the cell below gives the resistivity it implies.

In [ ]:
for t_um in (0.14, 0.20):
    rho = R_MEAS_7U5 * 7.5e-6 * t_um * 1e-6 / (LENGTH * 1e-6)  # ohm m
    print(
        f"7.5 x {t_um} um TiN, {R_MEAS_7U5:.0f} ohm over {LENGTH:g} um: "
        f"{rho * 1e8:.0f} uOhm cm (Table 3: 43 uOhm cm)"
    )

## Convergence and sensitivity

Each row below changes one model parameter of the 2.0 µm heater and re-solves. The Window, depth and boundary rows use
`precision="fast"` and are compared with a fast baseline. The precision rows are compared with the default solve above.

In [ ]:
base_fast = make_sim(precision="fast").solve(wavelength_um=WAVELENGTH).p_pi_mw
depth_18 = region_depth()

cases = {
    "Window ±30 µm (×1.5)": (dict(window=(-30.0, 30.0)), "fast", base_fast),
    "region 12 µm high": (dict(depth=depth_18 - 6.0), "fast", base_fast),
    "region 30 µm high": (dict(depth=depth_18 + 12.0), "fast", base_fast),
    "region 60 µm high": (dict(depth=depth_18 + 42.0), "fast", base_fast),
    "adiabatic top (h = 0)": (dict(top="adiabatic"), "fast", base_fast),
    'precision="fast"': (dict(), "fast", result_2u0.p_pi_mw),
    'precision="accurate"': (dict(), "accurate", result_2u0.p_pi_mw),
}
rows = []
for label, (kwargs, precision, reference) in cases.items():
    sim = make_sim(precision=precision, **kwargs)
    p_pi = sim.solve(wavelength_um=WAVELENGTH).p_pi_mw
    rows.append(
        {
            "change": label,
            "P_pi (mW)": round(p_pi, 2),
            "vs reference": f"{p_pi / reference - 1:+.2%}",
        }
    )
pd.DataFrame(rows).set_index("change")

- **Window and mesh are converged.** Growing the Window by half, or refining the mesh, changes P_π by well under 1 %.
- **Convection at the top is negligible.** h = 10 W/m²/K carries almost no heat next to the substrate path.
- **The region height is not a convergence parameter in 2D.** The more substrate we model, the hotter the device runs,
  so P_π drops slowly. A 2D section has no steady state without a sink at a finite distance, and heat that reaches the
  substrate spreads only sideways. In the real chip it also spreads along the 320 µm heater, which a 2D section cannot
  capture. We use the paper's 18 µm, which is what makes the comparison with its simulation like-for-like.

## Crosstalk onto a neighbouring waveguide

A second strip at a distance `gap` from the 2.0 µm heater's waveguide picks up some phase when the heater drives P_π.
Above an ideal heat sink, the closed form for a heater on a layer of thickness H predicts that the far-field temperature
decays as exp(−πx / 2H). That is a decay length of 2H/π ≈ 3.1 µm for H = 4.86 µm.

We solve the device twice:

- with the paper's region, 13 µm of Si substrate under the BOX;
- with an almost ideal sink, a 50 nm substrate held at ambient right under the BOX.

That separates the oxide's own decay from heat that travels through the substrate. The Window extends 20 µm beyond the
neighbour.

In [ ]:
gaps = [5.0, 10.0, 15.0, 20.0, 30.0]
substrates = {"paper's region (13 µm Si)": None, "ideal sink under the BOX": 0.05}
crosstalk = {label: [] for label in substrates}
for gap in gaps:
    for label, depth in substrates.items():
        s = make_sim(
            window=(-20.0, gap + 20.0),
            depth=depth,
            neighbour_gap=gap,
            precision="fast",
        )
        r = s.solve(wavelength_um=WAVELENGTH)
        crosstalk[label].append(r.crosstalk_rad["neighbour"])

decay_length = 2 * H_OXIDE / math.pi
anchor = crosstalk["ideal sink under the BOX"][1]  # match the closed form at 10 um
analytic = anchor * np.exp(-(np.array(gaps) - gaps[1]) / decay_length)

fig, ax = plt.subplots()
for label, values in crosstalk.items():
    ax.semilogy(gaps, values, "o-", label=label)
ax.semilogy(
    gaps, analytic, "k--", label=f"exp(-x / {decay_length:.2f} µm), closed form"
)
ax.set_xlabel("Gap between waveguide centres (µm)")
ax.set_ylabel("Neighbour phase at P_π (rad)")
ax.set_title("Thermal crosstalk")
ax.legend()

pd.DataFrame(crosstalk, index=pd.Index(gaps, name="gap (µm)")).assign(
    **{"closed form, ideal sink": analytic}
).map(lambda v: f"{v:.2e}")

With an ideal sink under the BOX, the crosstalk follows the closed-form exponential decay over more than three decades.
That is the check on the solver. With the paper's 13 µm of Si substrate the decay slows beyond about 10 µm. Heat that
reaches the substrate spreads sideways and warms the oxide under the neighbour. At 20 µm the crosstalk is about twice
the ideal-sink value, and at 30 µm about ten times, so the substrate path dominates the far field. Measured crosstalk on
SOI shows the same slow, power-law-like tail: Harris et al. 2014 fit an exponent of about 0.94. In 2D, the size of that
tail depends on the region height, for the same reason P_π does. Treat far-field crosstalk from a 2D model as an
order-of-magnitude estimate.

## Summary

Every comparison against the paper, with its tolerance. The error plot shows how far each lands from its target, and
the shaded band is the tolerance.

In [ ]:
lo_band, hi_band = 0.85 * P_PI_MEAS_7U5[0], 1.15 * P_PI_MEAS_7U5[1]
tau_mid = 0.5 * (TAU_RISE_7U5 + TAU_FALL_7U5)
summary = pd.DataFrame(
    [
        {
            "comparison": "P_pi, 2.0 µm vs Lumerical",
            "model": f"{result_2u0.p_pi_mw:.2f} mW",
            "target": f"{P_PI_SIM_2U0[0]}–{P_PI_SIM_2U0[1]} mW",
            "error": outside(result_2u0.p_pi_mw, P_PI_SIM_2U0),
            "tolerance": 0.0,
        },
        {
            "comparison": "tau_1e, 2.5 µm vs Lumerical",
            "model": f"{step_2u5.tau_1e_us:.2f} µs",
            "target": f"{TAU_SIM_2U5} µs ±20 %",
            "error": step_2u5.tau_1e_us / TAU_SIM_2U5 - 1,
            "tolerance": 0.20,
        },
        {
            "comparison": "P_pi, 7.5 µm vs measured",
            "model": f"{result_7u5.p_pi_mw:.2f} mW",
            "target": f"{P_PI_MEAS_7U5[0]}–{P_PI_MEAS_7U5[1]} mW ±15 %",
            "error": outside(result_7u5.p_pi_mw, (lo_band, hi_band)),
            "tolerance": 0.0,
        },
        {
            "comparison": "tau_1e, 7.5 µm vs measured",
            "model": f"{step_7u5.tau_1e_us:.2f} µs",
            "target": f"between {TAU_RISE_7U5} and {TAU_FALL_7U5} µs",
            "error": max(0.0, step_7u5.tau_1e_us / TAU_FALL_7U5 - 1)
            + min(0.0, step_7u5.tau_1e_us / TAU_RISE_7U5 - 1),
            "tolerance": 0.0,
        },
    ]
).set_index("comparison")
summary["verdict"] = np.where(
    summary["error"].abs() <= summary["tolerance"] + 1e-12, "pass", "miss"
)

fig, ax = plt.subplots(figsize=(7, 3))
y = np.arange(len(summary))
ax.barh(
    y,
    100 * summary["error"],
    color=["C0" if v == "pass" else "C3" for v in summary["verdict"]],
)
for yi, tol in zip(y, summary["tolerance"]):
    ax.fill_betweenx(
        [yi - 0.4, yi + 0.4], -100 * tol, 100 * tol, color="0.85", zorder=0
    )
ax.plot(100 * summary["error"], y, "o", color="0.2")
ax.axvline(0, color="0.3", lw=0.8)
ax.set_yticks(y, summary.index)
ax.invert_yaxis()
ax.set_xlabel("Error against the target (%)")
ax.set_title("gsim against Jacques et al. 2019")
fig.tight_layout()

summary.assign(error=summary["error"].map(lambda e: f"{e:+.1%}"))[
    ["model", "target", "error", "verdict"]
]

For the banded comparisons the error is the distance outside the band (for the measured P_π, the band plus a ±15 %
margin), so zero means inside. R, about 0.13 kΩ here against the measured 0.54 kΩ total, is reported but not compared.

## Assumptions and limits of the 2D model

- **Infinitely long heater.** No heater ends, contacts or via spreading. P_π does not depend on the heater length in 2D.
  The length sets only R, V_π and the phase at a given power.
- **Linear.** k, ρc_p and dn/dT do not depend on temperature, and the heater's resistance does not change as it heats.
  So the rise and the fall have the same time constant. The heater runs tens of kelvin above ambient at P_π, and Si's
  dn/dT changes by a few percent over that range.
- **Uniform Joule heating.** The current runs along the waveguide with uniform density in the TiN. There is no
  free-carrier or optical-loss change with temperature.
- **Substrate.** The region height is a model choice, not a converged parameter. We use the paper's 18 µm. It shifts
  P_π by a few percent and sets the far-field crosstalk.
- **Withheld inputs.** The TiN thickness (0.14 µm) and the oxide above the heater (0.5 µm) are assumptions. The
  sensitivity table shows what each moves.

## Next

- [Thermal guide](../api/thermal.md): the setup, the numerical knobs and the results (`Simulation`, `ThermalResult`,
  `StepResponse`, `HeaterReport`), with links to the API reference.